# B7q · primer: what the docking study of 82 nitrosamines can and cannot show
**Series B, Post 7q** &nbsp;|&nbsp; companion *primer* notebook to the docking notebook (the 82 nitrosamines in CYP2A13, PDB 4EJH)

The docking notebook asks whether the **Vina score** or the **geometry for activation** (the distance of a nitrosamine's α-carbon to the heme iron) tracks the carcinogenic potency pTD$_{50}$ of 82 nitrosamines. All five tests of the family are null. This primer does not repeat the docking (that needs AutoDock Vina, Meeko, Molscrub and reduce2, about thirty minutes of compute). It rebuilds and **checks what surrounds the docking**, in plain Python with no chemistry package: the **82-molecule table** from the raw tables, with an RDKit-free SMILES parser; the **α-carbons and α-hydrogens** from the molecular graph, validated against the $86$ α-hydrogen counts of the CPCA table; the **statistics** of the five tests (permutation $p$-values, bootstrap intervals, the size-removed partial rank correlation) and a check of the printed results against closed-form approximations; what $82$ molecules can **detect** at all (power); a simulation of **confounding by size**; the **reliability of the reactive fraction**, which is a count out of nine poses; and a **cluster-split cross-validation** with noise-feature controls. Part II runs the real libraries: scipy, **RDKit** (the notebook's own table-building route, which settles two small differences), the **crystal structure 4EJH** (the six NNK copies, their α-carbon and iron distances) and, if you have its results file, the notebook's own five tests recomputed.

> **Two parts.** **Part I** needs only NumPy, SciPy, pandas and matplotlib and takes about a minute on a CPU. **Part II** needs RDKit, scikit-learn and a network connection (the structure comes from the Protein Data Bank); each stage is guarded.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | the 82 nitrosamines, rebuilt without RDKit |
| 2 | pTD$_{50}$: units, ranks and what the table says |
| 3 | α-carbons and α-hydrogens from the graph (validated on the CPCA table) |
| 4 | the five tests: statistics, printed results, power, confounding |
| 5 | the reactive fraction is a count out of nine |
| 6 | cluster-split cross-validation and noise-feature controls |
| 7 | reading the validation of the docking protocol |
| **Part II** | scipy, RDKit, the crystal structure 4EJH, the results file |

## 0. Setup

In [ ]:
import math, time, itertools, importlib.util, os, sys, re, io, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")

## 1. The 82 nitrosamines, rebuilt without RDKit
The docking notebook embeds three tables: $86$ carcinogenic nitrosamines with TD$_{50}$ values from four sources (the Carcinogenic Potency Database `cpdb`, the Lhasa database `lcdb`, a 2019 compilation `s19`, a `gold` set), $23$ non-carcinogens, and the $86$-row table of the CPCA (carcinogenic potency categorization approach), which gives for each compound the numbers of hydrogens on the carbons next to the N-nitroso group. The notebook turns them into $82$ unique compounds with RDKit: it keeps the largest fragment, tests for a nitrosamine (not a nitrosamide) with a SMARTS pattern, merges molecules with the same *non-stereo canonical SMILES* (taking the geometric mean of their TD$_{50}$) and takes the first available value in the order cpdb, lcdb, s19, gold.
We do the same with a small SMILES parser and a graph invariant that ignores stereochemistry, bond orders and the position of hydrogens, so that Kekulé and aromatic spellings and tautomers of one molecule agree.
**Stop and predict:** the raw table has $86$ positive rows. How many unique molecules remain, and which kinds of rows are merged?

In [ ]:
MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "Br": 79.904}
Z = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "S": 16, "Cl": 17, "Br": 35}
VAL = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1, "Br": 1}
TOKEN = re.compile(r"\[[^\]]+\]|Cl|Br|[BCNOSPFI]|[bcnosp]|\(|\)|=|#|-|:|\/|\\|\d|%\d\d|\.")
def parse_smiles(smi):
    """SMILES -> atoms (element, aromatic, charge, explicit_H or None) and bonds (i, j, order); order 1.5 marks an aromatic bond"""
    atoms, bonds, stack, prev, pending, rings = [], [], [], None, None, {}
    for tok in TOKEN.findall(smi):
        if tok == "(": stack.append(prev)
        elif tok == ")": prev = stack.pop()
        elif tok in "-=#:": pending = {"-": 1, "=": 2, "#": 3, ":": 1.5}[tok]
        elif tok.isdigit():
            if tok in rings: j, o = rings.pop(tok); bonds.append((prev, j, o if o is not None else (pending or (1.5 if atoms[prev][1] and atoms[j][1] else 1)))); pending = None
            else: rings[tok] = (prev, pending); pending = None
        else:
            if tok.startswith("["):
                m = re.match(r"\[([A-Za-z][a-z]?)(H\d?)?([+-]\d?)?\]", tok); el = m.group(1); arom = el.islower(); el = el.capitalize()
                nh = 0 if not m.group(2) else (1 if m.group(2) == "H" else int(m.group(2)[1:])); ch = 0 if not m.group(3) else (int(m.group(3)[1:] or 1) * (1 if m.group(3)[0] == "+" else -1)); atoms.append((el, arom, ch, nh))
            else: arom = tok.islower(); atoms.append((tok.capitalize(), arom, 0, None))
            i = len(atoms) - 1
            if prev is not None: bonds.append((prev, i, pending if pending else (1.5 if atoms[prev][1] and arom else 1)))
            prev, pending = i, None
    return atoms, bonds
def add_hydrogens(atoms, bonds):
    """implicit hydrogens from the default valences (aromatic bonds count 1, plus one pi bond for carbon and pyridine-type nitrogen)"""
    used = [0.0] * len(atoms); nb = [[] for _ in atoms]
    for i, j, o in bonds: nb[i].append((j, o)); nb[j].append((i, o)); w = 1 if o == 1.5 else o; used[i] += w; used[j] += w
    H = []
    for i, (el, arom, ch, nh) in enumerate(atoms):
        if nh is not None: H.append(nh); continue                                             # bracket atoms: the hydrogens are written
        v = VAL[el] + (1 if (el == "N" and ch == 1) else 0) - (1 if (el == "O" and ch == 1) else 0)
        u = used[i]
        if arom and el in ("C", "N") and u < v: u += 1                                          # the pi bond of an aromatic atom
        H.append(int(max(0, v - u)))
    return H, nb
def ring_atoms(n, bonds):
    """atoms and bonds that lie on a ring: a bond is a ring bond if its ends stay connected without it"""
    adj = {i: [] for i in range(n)}
    for k, (i, j, o) in enumerate(bonds): adj[i].append((j, k)); adj[j].append((i, k))
    def connected(a, b, skip):
        seen, st = {a}, [a]
        while st:
            x = st.pop()
            if x == b: return True
            for y, k in adj[x]:
                if k != skip and y not in seen: seen.add(y); st.append(y)
        return False
    rb = [connected(i, j, k) for k, (i, j, o) in enumerate(bonds)]; ra = set()
    for k, (i, j, o) in enumerate(bonds):
        if rb[k]: ra.update((i, j))
    return ra, rb
def mol_weight(atoms, H): return sum(MASS[a[0]] for a in atoms) + sum(H) * MASS["H"]
def hash32(values):
    h = 2166136261
    for v in values: h = ((h ^ (int(v) & 0xFFFFFFFF)) * 16777619) & 0xFFFFFFFF
    return h
def circular_environments(smi, radius=2, salt=0):
    """Morgan-style circular fingerprint: the list of distinct environment identifiers (before folding); the duplicate rule drops an environment that covers the same bonds as an earlier one"""
    atoms, bonds = parse_smiles(smi); H, nb = add_hydrogens(atoms, bonds); ra, rb = ring_atoms(len(atoms), bonds); n = len(atoms)
    inv = [hash32([salt, Z[a[0]], len(nb[i]), H[i], a[2], int(i in ra)]) for i, a in enumerate(atoms)]; ids = set(inv); seen = [tuple()]; dead = [False] * n
    bset = [frozenset() for _ in range(n)]; bidx = {}
    for k, (i, j, o) in enumerate(bonds): bidx[(i, j)] = bidx[(j, i)] = k
    for layer in range(1, radius + 1):
        new_inv, new_bs, cand = list(inv), list(bset), []
        for i in range(n):
            if dead[i]: continue
            ns = sorted((int(o * 2), inv[j]) for j, o in nb[i]); new_inv[i] = hash32([salt, layer, inv[i]] + [x for p in ns for x in p])
            s = set(bset[i])
            for j, o in nb[i]: s |= set(bset[j]); s.add(bidx[(i, j)])
            new_bs[i] = frozenset(s); cand.append((tuple(sorted(new_bs[i])), new_inv[i], i))
        for bs, iv, i in sorted(cand):
            if bs not in seen: ids.add(iv); seen.append(bs)
            else: dead[i] = True
        inv, bset = new_inv, new_bs
    return sorted(ids)
def fold(env_ids, size=1024): x = np.zeros(size); x[[e % size for e in env_ids]] = 1; return x

In [ ]:
def clean(smi):
    """remove stereo marks and explicit [H] atoms: the topology is unchanged"""
    s = re.sub(r"\[([A-Za-z][a-z]?)@+(H?\d?)", lambda m: "[" + m.group(1) + m.group(2), smi); s = s.replace("/", "").replace("\\", ""); return s.replace("([H])", "").replace("[H]", "")
class Mol:
    def __init__(self, smi):
        parts = [p for p in clean(smi).split(".") if p]; best = max(parts, key=lambda p: sum(c.isalpha() and c.isupper() for c in p)); self.smi = best
        self.at, self.bd = parse_smiles(best); self.H, self.nb = add_hydrogens(self.at, self.bd); self.n = len(self.at); self.el = [a[0] for a in self.at]
        self.ra, self.rb = ring_atoms(self.n, self.bd)
    def mw(self): return mol_weight(self.at, self.H)
    def heavy(self): return self.n
    def deg(self, i): return len(self.nb[i])
    def nbrs(self, i): return [j for j, o in self.nb[i]]
    def order(self, i, j): return next(o for k, o in self.nb[i] if k == j)
    def nitroso_groups(self):
        """(amine N, nitroso N, O) for every N-N=O; the amine N has exactly 3 neighbours"""
        out = []
        for n2 in range(self.n):
            if self.el[n2] != "N": continue
            for o in self.nbrs(n2):
                if self.el[o] == "O" and self.order(n2, o) == 2 and self.deg(n2) == 2 and self.deg(o) == 1:
                    for n1 in self.nbrs(n2):
                        if self.el[n1] == "N" and n1 != o and self.deg(n1) == 3: out.append((n1, n2, o))
        return out
    def is_nitrosamine(self):
        """[#6][NX3;!$(N-[#6]=[O,S]);!$(N-S(=O)=O)](-[#6])-N=O and not acyl-nitroso"""
        ok = False
        for n1, n2, o in self.nitroso_groups():
            cs = [j for j in self.nbrs(n1) if j != n2]
            if len(cs) != 2 or not all(self.el[c] == "C" for c in cs): continue
            acyl = any(self.el[c] == "C" and any(self.el[k] in "OS" and k != n1 and self.order(c, k) == 2 for k in self.nbrs(c)) for c in cs) or any(self.el[k] == "S" and any(self.el[x] == "O" and self.order(k, x) == 2 for x in self.nbrs(k)) for k in self.nbrs(n1) if k != n2)
            if not acyl: ok = True
        return ok
    def alpha_carbons(self):
        a = []
        for n1, n2, o in self.nitroso_groups(): a += [j for j in self.nbrs(n1) if j != n2 and self.el[j] == "C"]
        return a
    def key(self, rounds=4):
        """graph invariant that ignores bond orders and aromatic flags (element, charge, degree) plus the total number of hydrogens, so that Kekule and aromatic spellings and tautomers of one molecule agree"""
        lab = {i: (self.el[i], self.at[i][2], self.deg(i)) for i in range(self.n)}
        for _ in range(rounds): lab = {i: hash((lab[i], tuple(sorted(lab[j] for j, o in self.nb[i])))) for i in range(self.n)}
        return (self.n, int(sum(self.H)), hash(tuple(sorted(lab.values()))))

In [ ]:
POS_CSV = r"""name,cas,smiles,cpdb,lcdb,s19,gold
N-Nitroso-methyl-2-phenylethylamine (NMPEA),13256-11-6,CN(CCC1=CC=CC=C1)N=O,0.00998,0.00797,,0.00998
N-Nitrosomethyl(2-oxopropyl)amine,55984-51-5,CC(=O)CN(C)N=O,0.0172,0.0175,0.017,0.0172
"N-Nitrosodiethylamine (DEN, NDEA)",55-18-5,CCN(CC)N=O,0.0265,0.0177,,0.0265
"N-Nitroso(2,3-dihydroxypropyl)(2-oxopropyl)amine",92177-50-9,CC(=O)CN(CC(O)CO)N=O,0.0352,,0.0352,0.0352
N-Nitrosoheptamethyleneimine,20917-49-1,O=NN1CCCCCCC1,0.0378,0.0383,,0.0378
N-Nitrosomethyl-2-hydroxypropylamine,75411-83-5,CC(O)CN(C)N=O,0.0463,,0.0442,0.0463
N-Nitroso-methylethylamine,10595-95-6,CCN(C)N=O,0.0503,,,0.0503
"N-Nitroso-2,3-dihydroxypropyl-2-hydroxypropylamine",89911-79-5,CC(O)CN(CC(O)CO)N=O,0.0535,,,0.0535
"N-Nitroso-1,2,3,6-tetrahydropyridine (NTHP)",55556-92-8,O=NN1CCC=CC1,0.0601,0.0599,,0.0601
Dinitrosohomopiperazine,55557-00-1,O=NN1CCCN(N=O)CC1,0.0615,0.134,0.242,
N′-Nitrosonornicotine,53759-22-1,O=NN1CCCC1c1cccnc1,,,0.0957,0.0957
"N-Nitrosodimethylamine (DMN, NDMA)",62-75-9,CN(C)N=O,0.0959,0.177,,0.0959
4-(Methylnitrosamino)-1-(3-pyridyl)-1-butanone (NNK),64091-91-4,CN(CCCC(=O)C1=CN=CC=C1)N=O,0.142,0.0999,,0.0999
4-(Methylnitrosoamino)-1-(3-pyridyl)-1-butanol,76014-81-8,CN(CCCC(O)C1=CC=CN=C1)N=O,0.103,,,0.103
N-Methyl-N-nitrosoaniline,614-00-6,CN(N=O)c1ccccc1,0.142,0.106,0.106,0.142
N-Nitrosomorpholine,59-89-2,O=NN1CCOCC1,0.109,0.135,0.129,0.109
N-Methyl-N′-nitrosopiperazine,16339-07-4,CN1CCN(N=O)CC1,,,0.14,
"1-Nitroso-3,4,5-trimethylpiperazine",75881-18-4,CC1CN(N=O)CC(C)N1C,0.151,0.153,0.153,0.151
"N,N'-Dinitrosohexahydropyrimidine",15973-99-6,O=NN1CCCN(C1)N=O,0.166,0.168,,
N-Nitrosobis(2-oxopropyl)amine,60599-38-4,CC(=O)CN(CC(C)=O)N=O,0.491,0.184,0.286,0.491
Dinitrosocaffeidine,145438-97-7,CN(N=O)C(=O)c1c(N(C)N=O)ncn1C,,,0.185,
N-Nitroso-di-n-propylamine (DPN/NDPA),621-64-7,CCCN(CCC)N=O,0.186,,,0.186
2-Nitrosomethylaminopyridine,16219-98-0,CN(N=O)C1=NC=CC=C1,0.214,,,0.214
N-Nitroso-N-methyl-4-fluoroaniline,937-25-7,CN(N=O)c1ccc(F)cc1,0.255,,0.255,0.255
4-(Butylnitrosoamino)-1-butanol,3817-11-6,CCCCN(CCCCO)N=O,0.457,0.261,,0.457
N-Nitrosohexamethyleneimine,932-83-2,O=NN1CCCCCC1,,,0.313,0.528
N-Nitrosoallyl-2-oxopropylamine,91308-71-3,C=CCN(CC(C)=O)N=O,0.335,,0.335,0.335
4-(4-N-Methyl-N-nitrosaminostyryl)quinoline,16699-10-8,CN(N=O)c1ccc(/C=C/c2ccnc3ccccc23)cc1,,,0.468,0.699
Name not provided in source,81795-07-5,N1(C(SC(SC1C)C)C)N=O,,,,0.483
2-(Nitroso-2-propenylamino)ethanol,91308-69-9,OCCN(CC=C)N=O,0.491,,,0.491
N-Nitroso-4-piperidone,55556-91-7,O=NN1CCC(=O)CC1,,,0.499,
Methyldodecylnitrosamine,55090-44-3,CCCCCCCCCCCCN(C)N=O,0.537,,,0.537
N′-Nitrosonornicotine-1-N-oxide,78246-24-9,O=NN1CCCC1c1ccc[n+]([O-])c1,,,0.573,0.876
N-Nitroso-4-piperidinol,55556-93-9,O=NN1CCC(O)CC1,,,0.596,
"N-Nitrosomethyl-2,3-dihydroxypropylamine",86451-37-8,CN(CC(O)CO)N=O,0.646,,0.646,0.646
3-Methyl-1-nitrosopiperidine,13603-07-1,CC1CCCN(N=O)C1,,,0.665,
4-Methyl-1-nitrosopiperidine,15104-03-7,CC1CCN(N=O)CC1,,,0.665,
N-Nitrosopyrrolidine,930-55-2,O=NN1CCCC1,0.799,2.02,2.47,0.679
N-Nitroso-di-n-butylamine (DBN/NDBA),924-16-3,CCCCN(CCCC)N=O,0.691,,,0.691
"N-Nitrosobis (4,4,4-trifluorobutyl) amine",83335-32-4,FC(F)(F)CCCN(CCCC(F)(F)F)N=O,0.748,,,0.748
3-Nitrosooxazolidine,39884-52-1,O=NN1CCOC1,0.798,,,0.798
3-Hydroxy-N-nitrosopiperidine,55556-85-9,O=NN1CCCC(O)C1,,,0.819,
"N-Nitrosoallyl-2,3-dihydroxypropylamine",88208-16-6,C=CCN(CC(O)CO)N=O,0.825,,0.825,0.825
N-Nitrosobis(2-hydroxypropyl)amine,53609-64-6,CC(O)CN(CC(C)O)N=O,0.846,1.78,0.891,0.846
"1-Nitroso-3,5-dimethylpiperazine",67774-31-6,CC1CN(N=O)CC(C)N1,,,0.866,
1-(Nitroso-2-propenylamino)-2-propanol,91308-70-2,CC(O)CN(CC=C)N=O,0.877,,,0.877
"1-Nitroso-4-acetyl-3,5-dimethylpiperazine",75881-17-3,CC(=O)N1C(C)CN(N=O)CC1C,,,0.921,
N-Nitrosopiperidine,100-75-4,O=NN1CCCCC1,,1.12,0.974,1.3
Nitrosomethyl-3-carboxypropylamine,61445-55-4,CN(CCCC(O)=O)N=O,0.982,,,0.982
N-Nitroso(2-hydroxyethyl)(2-oxopropyl)amine,92177-49-6,CC(=O)CN(CCO)N=O,1.8,,0.997,0.997
Mononitrosocaffeidine,145438-96-6,CNC(=O)c1c(N(C)N=O)ncn1C,1.01,1.0,1.01,1.0
N-Nitroso(2-hydroxypropyl)(2-hydroxyethyl)amine,75896-33-2,CC(O)CN(CCO)N=O,1.02,,1.02,1.02
N-Nitroso(2-hydroxypropyl)(2-oxopropyl)amine (HPOP),61499-28-3,CC(=O)CN(CC(C)O)N=O,,,1.1,
"N-Nitroso-2,6-dimethylmorpholine",1456-28-6,CC1CN(N=O)CC(C)O1,,,1.22,2.0
N-Methyl-N-nitroso-1-decanamine,75881-22-0,CCCCCCCCCCN(C)N=O,1.26,,,1.26
N-Methyl-N-nitrosoethanolamine,26921-68-6,CN(CCO)N=O,1.29,,,1.29
"N-Methyl-N,4-dinitrosoaniline",99-80-9,CN(N=O)c1ccc(N=O)cc1,1.3,,1.3,1.3
"N-Nitroso-N,N-diallylamine",16338-97-9,C=CCN(CC=C)N=O,,,,1.54
N-Nitroso-N-methyltetradecylamine,75881-20-8,CCCCCCCCCCCCCCN(C)N=O,1.65,,,1.65
3-(Methylnitrosoamino)-1-propanol,70415-59-7,CN(CCCO)N=O,1.66,,,1.66
N-Methyl-N-nitroso-1-undecanamine,68107-26-6,CCCCCCCCCCCN(C)N=O,2.37,,,2.37
"N-Ethyl-2,2,2-trifluoro-N-nitrosoethanamine",82018-90-4,CCN(CC(F)(F)F)N=O,2.52,2.55,,2.52
"1,4-Dinitroso-2,6-dimethylpiperazine",55380-34-2,CC1CN(N=O)CC(C)N1N=O,3.1,,3.1,
N-Nitrosodiethanolamine (NDELA),1116-54-7,OCCN(CCO)N=O,3.17,3.38,,3.17
4-Nitrosothiomorpholine,26541-51-5,O=NN1CCSCC1,5.39,3.54,,5.39
"1,4-Dinitrosopiperazine",140-79-4,O=NN1CCN(N=O)CC1,3.6,,8.7,
"2-(Methylnitrosamino)-N,N-dimethylethylamine",23834-30-2,CN(C)CCN(C)N=O,3.83,,,3.83
"N,N-Dipentylnitrosamine",13256-06-9,CCCCCN(CCCCC)N=O,4.03,4.09,,4.03
N′-Nitrosoanabasine (NAB),37620-20-5,O=NN1CCCCC1c1cccnc1,,,4.14,
Name not provided in source,66398-63-8,O=S(=O)(C=1C=CC(=CC=1)C)OCCN(C)N=O,,,,4.8
"N-Nitroso-2,3-dihydroxypropylethanolamine",89911-78-4,O=NN(CCO)CC(O)CO,5.98,6.04,6.04,5.98
1-Nitrosoazetidine,15216-10-1,O=NN1CCC1,7.14,,,7.14
N-Nitroso-3-hydroxypyrrolidine,56222-35-6,O=NN1CCC(O)C1,,,8.11,7.65
N-Nitrosopiperazine,5632-47-3,O=NN1CCNCC1,,,34.6,8.78
"1-Nitroso-3,5-dimethyl-4-benzoylpiperazine",61034-40-0,CC1CN(N=O)CC(C)N1C(=O)c1ccccc1,,,9.1,9.66
Name not provided in source,40580-89-0,C1CCN(CCCCCCCCC1)N=O,,,,10.9
N-Nitrosoanabasine,1133-64-8,O=NN1CCCC[C@H]1C2=CN=CC=C2,11.9,,,11.9
S(+)-2-Methyl-N-nitrosopiperidine,36702-44-0,C[C@H]1CCCCN1N=O,,,49.4,13.2
N⁶-(Methylnitroso)adenosine,Not assigned in SI,CN(N=O)c1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O,,,15.8,
Name not provided in source,NoCAS-2163,C1(=NC=NC2=C1NC=N2)N(C)N=O,,,,18.0
N⁶-(Methylnitroso)adenine,21928-82-5,CN(N=O)c1ncnc2[nH]cnc12,,,18.1,18.3
R(−)-2-Methyl-N-nitrosopiperidine,14026-03-0,C[C@@H]1CCCCN1N=O,,,22.1,20.4
"2,5-Dimethyl-N-nitrosopyrrolidine",55556-86-0,CC1CCC(C)N1N=O,,,31.3,
Name not provided in source,17608-59-2,C1=CC=CC(=C1)C(C(N(N=O)C)C)O,,,,95.2
N-Nitroso-(pseudo)ephedrine (NEP/NPEP),17608-59-2,OC(C(N(N=O)C)(C)[H])(C(=CC=C1)C=C1)[H],95.2,,,
N-Nitroso-diphenylamine (NDPh),86-30-6,O=NN(C1=CC=CC=C1)C2=CC=CC=C2,,,,167.0
"""
NEG_CSV = r"""name,cas,smiles,src
1-Nitroso-L-proline,7519-36-0,OC(=O)[C@@H]1CCCN1N=O,CPCA paper (CPDB/other negative)
4-Nitrosomethylaminopyridine,16219-99-1,CN(N=O)C1=CC=NC=C1,CPCA paper (CPDB/other negative)
"N,N'-Dinitrosopentamethylenetetramine",101-25-7,O=NN1CN2CN(C1)CN(C2)N=O,CPCA paper (CPDB/other negative)
"N,N'-Dinitrosopentamethylenetetramine",101-25-7,O=NN1CN2CN(C1)CN(C2)N=O,CPCA paper (CPDB/other negative)
"N-Nitrosobis(2,2,2-trifluoroethyl)amine",625-89-8,FC(F)(F)CN(CC(F)(F)F)N=O,CPCA paper (CPDB/other negative)
N-nitroso-di-secondary-butylamine ,5350-17-4,CCC(C)N(N=O)C(C)CC,CPCA paper (CPDB/other negative)
N-Nitrosodithiazine,114282-83-6,O=NN1CSCSC1,CPCA paper (CPDB/other negative)
N-Nitrosoiminodiacetic acid,25081-31-6,OC(=O)CN(CC(O)=O)N=O,CPCA paper (CPDB/other negative)
N-Nitroso-N-methyl-3-aminopyridine,69658-91-9,CN(N=O)C1=CC=CN=C1,CPCA paper (CPDB/other negative)
N-Nitrosopipecolinic acid,4515-18-8,OC(=O)C1CCCCN1N=O,CPCA paper (CPDB/other negative)
N-Nitroso-tert-butylmethylamine ,2504-18-9,CN(N=O)C(C)(C)C,CPCA paper (CPDB/other negative)
114282-83-6,114282-83-6,C1SCSCN1N=O,Dobo 2022 (LCDB call)
7519-36-0,7519-36-0,[C@H]1(C(O)=O)CCCN1N=O,Dobo 2022 (LCDB call)
4515-18-8,4515-18-8,C1CCN(C(C1)C(O)=O)N=O,Dobo 2022 (LCDB call)
16219-99-1,16219-99-1,C1=CC(=CC=N1)N(C)N=O,Dobo 2022 (LCDB call)
55557-03-4,55557-03-4,C(C(OC)=O)(C1N(CCCC1)N=O)C2=CC=CC=C2,Dobo 2022 (LCDB call)
943-41-9,943-41-9,C1=C(C=CC(=C1)N(N=O)C)[N+](=O)[O-],Dobo 2022 (LCDB call)
69658-91-9,69658-91-9,C1=CC=C(C=N1)N(C)N=O,Dobo 2022 (LCDB call)
25081-31-6,25081-31-6,C(N(CC(O)=O)N=O)C(O)=O,Dobo 2022 (LCDB call)
67587-52-4,67587-52-4,N1(N=O)CC(OCC1)O,Dobo 2022 (LCDB call)
30310-80-6,30310-80-6,C(=O)([C@@H]1C[C@H](CN1N=O)O)O,Dobo 2022 (LCDB call)
55557-02-3,55557-02-3,N1(CC/C=C(\\C1)C(OC)=O)N=O,Dobo 2022 (LCDB call)
625-89-8,625-89-8,C(C(F)(F)F)N(CC(F)(F)F)N=O,Dobo 2022 (LCDB call)
"""
CPCA_CSV = r"""cas,name,smiles,cat,score,aH,nno
91308-70-2,1-(Nitroso-2-propenylamino)-2-propanol,CC(O)CN(CC=C)N=O,2,2.0,"2,2",1
91308-71-3,1-(Nitroso-2-propenylamino)-2-propanone,CC(=O)CN(CC=C)N=O,1,1.0,"2,2",1
75881-18-4,"1,2,6-Trimethyl-4-nitrosopiperazine (TMNP)",CC1CN(CC(C)N1C)N=O,2,2.0,"2,2",1
140-79-4,"1,4-Dinitrosopiperazine",O=NN1CCN(CC1)N=O,3,3.0,"2,2",2
140-79-4,"1,4-Dinitrosopiperazine",O=NN1CCN(CC1)N=O,3,3.0,"2,2",1
75896-33-2,1-[(2-Hydroxyethyl)nitrosoamino]-2-propanol,CC(O)CN(CCO)N=O,3,3.0,"2,2",1
92177-49-6,1-[(2-Hydroxyethyl)nitrosoamino]-2-propanone,O=C(CN(N=O)CCO)C,2,2.0,"2,2",1
15216-10-1,1-Nitrosoazetidine,O=NN1CCC1,1,1.0,"2,2",1
7519-36-0,1-Nitroso-L-proline,OC(=O)[C@@H]1CCCN1N=O,4,9.0,"1,2",1
23834-30-2,"2-(Methylnitrosamino)-N,N-dimethylethylamine",CN(C)CCN(C)N=O,1,1.0,"2,3",1
91308-69-9,2-(Nitroso-2-propenylamino)ethanol,OCCN(CC=C)N=O,2,2.0,"2,2",1
55380-34-2,"2,6-Dimethyldinitrosopiperazine",CC1CN(CC(C)N1N=O)N=O,2,,"1,1",2
55380-34-2,"2,6-Dimethyldinitrosopiperazine",CC1CN(CC(C)N1N=O)N=O,2,2.0,"2,2",1
16219-98-0,2-Nitrosomethylaminopyridine,CN(N=O)C1=NC=CC=C1,2,2.0,"0,3",1
86451-37-8,"3-(Methylnitrosoamino)-1,2-propanediol",CN(CC(O)CO)N=O,2,2.0,"2,3",1
70415-59-7,3-(Methylnitrosoamino)-1-propanol,CN(CCCO)N=O,1,1.0,"2,3",1
89911-78-4,"3-[(2-Hydroxyethyl)nitrosoamino]-1,2-propanediol",OCCN(CC(O)CO)N=O,3,3.0,"2,2",1
39884-52-1,3-Nitrosooxazolidine,O=NN1CCOC1,3,3.0,"2,2",1
3817-11-6,4-(Butylnitrosoamino)-1-butanol,CCCCN(CCCCO)N=O,1,1.0,"2,2",1
64091-91-4,4-(Methylnitrosamino)-1-(3-pyridyl)-1-butanone (NNK) ,CN(CCCC(=O)C1=CN=CC=C1)N=O,1,1.0,"2,3",1
76014-81-8,4-(Methylnitrosoamino)-1-(3-pyridyl)-1-butanol,CN(CCCC(O)C1=CC=CN=C1)N=O,1,1.0,"2,3",1
16219-99-1,4-Nitrosomethylaminopyridine,CN(N=O)C1=CC=NC=C1,2,2.0,"0,3",1
26541-51-5,4-Nitrosothiomorpholine,O=NN1CCSCC1,4,4.0,"2,2",1
55557-00-1,Dinitrosohomopiperazine,O=NN1CCCN(CC1)N=O,2,2.0,"2,2",2
55557-00-1,Dinitrosohomopiperazine,O=NN1CCCN(CC1)N=O,2,2.0,"2,2",1
937-25-7,Methyl(4-fluorophenyl)nitrosamine,CN(N=O)C1=CC=C(F)C=C1,2,2.0,"0,3",1
55090-44-3,Methyldodecylnitrosamine,CCCCCCCCCCCCN(C)N=O,1,1.0,"2,3",1
145438-96-6,Mononitrosocaffeidine,CNC(=O)C1=C(N=CN1C)N(C)N=O,2,2.0,"0,3",1
99-80-9,"N,4-Dinitroso-N-methylaniline",CN(N=O)C1=CC=C(C=C1)N=O,2,2.0,"0,3",1
15973-99-6,"N,N'-Dinitrosohexahydropyrimidine",O=NN1CCCN(C1)N=O,3,3.0,"2,2",2
15973-99-6,"N,N'-Dinitrosohexahydropyrimidine",O=NN1CCCN(C1)N=O,3,3.0,"2,2",1
101-25-7,"N,N'-Dinitrosopentamethylenetetramine",O=NN1CN2CN(C1)CN(C2)N=O,3,3.0,"2,2",2
101-25-7,"N,N'-Dinitrosopentamethylenetetramine",O=NN1CN2CN(C1)CN(C2)N=O,3,3.0,"2,2",1
13256-06-9,"N,N-Dipentylnitrosamine",CCCCCN(CCCCC)N=O,2,2.0,"2,2",1
82018-90-4,"N-Ethyl-2,2,2-trifluoro-N-nitrosoethanamine",CCN(CC(F)(F)F)N=O,2,2.0,"2,2",1
13147-25-6,N-Ethyl-N-(2-hydroxyethyl)nitrosamine,CCN(CCO)N=O,2,2.0,"2,2",1
92177-50-9,"Nitroso-2,3-dihydroxypropyl-2-oxopropylamine",CC(=O)CN(CC(O)CO)N=O,2,2.0,"2,2",1
61445-55-4,Nitrosomethyl-3-carboxypropylamine,CN(CCCC(O)=O)N=O,4,4.0,"2,3",1
75881-22-0,N-Methyl-N-nitroso-1-decanamine,CCCCCCCCCCN(C)N=O,1,1.0,"2,3",1
68107-26-6,N-Methyl-N-nitroso-1-undecanamine,CCCCCCCCCCCN(C)N=O,1,1.0,"2,3",1
26921-68-6,N-Methyl-N-nitrosoethanolamine,CN(CCO)N=O,2,2.0,"2,3",1
33868-17-6,N-Nitroso(methyl)cyanamide,CN(N=O)C#N,2,2.0,"0,3",1
17608-59-2; 1850-88-0,N-Nitroso-(pseudo)ephedrine (NEP/NPEP),OC(C(N(N=O)C)(C)[H])(C(=CC=C1)C=C1)[H],4,4.0,"1,3",1
55556-92-8,"N-Nitroso-1,2,3,6-tetrahydropyridine (NTHP) ",O=NN1CCC=CC1,3,3.0,"2,2",1
67856-67-1,"N-Nitroso-2,2-bis-(diethoxyethyl)amine",O(C(OCC)CN(N=O)CC(OCC)OCC)CC,2,2.0,"2,2",1
89911-79-5,"N-Nitroso-2,3-dihydroxypropyl-2-hydroxypropylamine ",CC(O)CN(CC(O)CO)N=O,3,3.0,"2,2",1
88208-16-6,"N-Nitroso-2,3-dihydroxypropylallylamine",OCC(O)CN(CC=C)N=O,2,2.0,"2,2",1
1133-64-8,N-Nitrosoanabasine,O=NN1CCCC[C@H]1C2=CN=CC=C2,4,4.0,"1,2",1
83335-32-4,"N-Nitrosobis (4,4,4-trifluorobutyl) amine",FC(F)(F)CCCN(CCCC(F)(F)F)N=O,2,2.0,"2,2",1
625-89-8,"N-Nitrosobis(2,2,2-trifluoroethyl)amine",FC(F)(F)CN(CC(F)(F)F)N=O,3,3.0,"2,2",1
67856-68-2,N-Nitroso-bis-(2-chloroethyl)amine,ClCCN(CCCl)N=O,1,1.0,"2,2",1
1116-82-1,N-Nitroso-bis-(2-cyanoethyl)amine,O=NN(CCC#N)CCC#N,1,1.0,"2,2",1
53609-64-6,N-Nitrosobis(2-hydroxypropyl)amine (BHP),CC(O)CN(CC(C)O)N=O,3,3.0,"2,2",1
67856-65-9,N-Nitroso-bis-(2-methoxyethyl)amine,O(CCN(N=O)CCOC)C,1,1.0,"2,2",1
60599-38-4,N-Nitrosobis(2-oxopropyl)amine (BOP/NOPA),CC(=O)CN(CC(C)=O)N=O,1,1.0,"2,2",1
1116-54-7,N-Nitrosodiethanolamine (NDELA) ,OCCN(CCO)N=O,3,3.0,"2,2",1
55-18-5,"N-Nitrosodiethylamine (DEN, NDEA) ",CCN(CC)N=O,1,1.0,"2,2",1
997-95-5,N-Nitrosodiisobutylamine,CC(C)CN(CC(C)C)N=O,1,0.0,"2,2",1
601-77-4,N-Nitroso-di-isopropylamine (DIPNA/NDIPA) ,CC(C)N(N=O)C(C)C,5,,"1,1",1
62-75-9,"N-Nitrosodimethylamine (DMN, NDMA)",CN(C)N=O,1,1.0,"3,3",1
924-16-3,N-Nitroso-di-n-butylamine (DBN/NDBA),CCCCN(CCCC)N=O,1,1.0,"2,2",1
6335-97-3,N-Nitroso-di-n-octylamine,CCCCCCCCN(CCCCCCCC)N=O,2,2.0,"2,2",1
621-64-7,N-Nitroso-di-n-propylamine (DPN/NDPA) ,CCCN(CCC)N=O,1,1.0,"2,2",1
86-30-6,N-Nitroso-diphenylamine (NDPh),O=NN(C1=CC=CC=C1)C2=CC=CC=C2,5,,"0,0",1
5350-17-4,N-nitroso-di-secondary-butylamine ,CCC(C)N(N=O)C(C)CC,5,,"1,1",1
114282-83-6,N-Nitrosodithiazine,O=NN1CSCSC1,4,4.0,"2,2",1
20917-49-1,N-Nitrosoheptamethyleneimine,O=NN1CCCCCCC1,1,1.0,"2,2",1
932-83-2,N-Nitrosohexamethyleneimine,O=NN1CCCCCC1,2,2.0,"2,2",1
25081-31-6,N-Nitrosoiminodiacetic acid,OC(=O)CN(CC(O)=O)N=O,4,4.0,"2,2",1
16339-18-7,N-Nitrosoiminodiacetonitrile,O=NN(CC#N)CC#N,3,3.0,"2,2",1
75411-83-5,N-Nitroso-methyl-2-hydroxypropylamine (MHP),CC(O)CN(C)N=O,2,2.0,"2,3",1
55984-51-5,N-Nitroso-methyl-2-oxopropylamine (MOP),CN(CC(C)=O)N=O,1,1.0,"2,3",1
13256-11-6,N-Nitroso-methyl-2-phenylethylamine (NMPEA),CN(CCC1=CC=CC=C1)N=O,1,1.0,"2,3",1
10595-95-6,N-Nitroso-methylethylamine,CCN(C)N=O,1,1.0,"2,3",1
30533-08-5,"N-Nitrosomethylisopropylamine (NMIPA) (as reported by Rao et al., 1979; however, reported as ethyl-iso-propylnitrosamine by Druckrey et al., 1967; see Materials and Methods section for more details) ",O=NN(C(C)C)C,3,3.0,"1,3",1
31820-22-1,N-Nitrosomethylneopentylamine (NMNA),CN(CC(C)(C)C)N=O,1,1.0,"2,3",1
924-46-9,N-Nitrosomethyl-n-propylamine ,CCCN(C)N=O,1,1.0,"2,3",1
59-89-2,N-Nitrosomorpholine (NMOR) ,O=NN1CCOCC1,2,2.0,"2,2",1
16338-97-9,"N-Nitroso-N,N-diallylamine",C=CCN(CC=C)N=O,1,1.0,"2,2",1
69658-91-9,N-Nitroso-N-methyl-3-aminopyridine,CN(N=O)C1=CC=CN=C1,2,2.0,"0,3",1
614-00-6,N-Nitroso-N-methylaniline (NMPA) ,CN(N=O)C1=CC=CC=C1,2,2.0,"0,3",1
75881-20-8,N-Nitroso-N-methyltetradecylamine,CCCCCCCCCCCCCCN(C)N=O,1,1.0,"2,3",1
4515-18-8,N-Nitrosopipecolinic acid,OC(=O)C1CCCCN1N=O,4,8.0,"1,2",1
100-75-4,N-Nitrosopiperidine (NPIP) ,O=NN1CCCCC1,3,3.0,"2,2",1
930-55-2,N-nitroso-pyrrolidine (NPYR) ,O=NN1CCCC1,4,4.0,"2,2",1
2504-18-9,N-Nitroso-tert-butylmethylamine ,CN(N=O)C(C)(C)C,5,,"0,3",1
"""
P_RAW = pd.read_csv(io.StringIO(POS_CSV), dtype={"cas": str}); N_RAW = pd.read_csv(io.StringIO(NEG_CSV), dtype={"cas": str}); C_RAW = pd.read_csv(io.StringIO(CPCA_CSV), dtype={"cas": str, "aH": str})
print("raw tables:", len(P_RAW), "positives,", len(N_RAW), "negatives,", len(C_RAW), "CPCA rows")

In [ ]:
num = lambda x: float(x) if pd.notna(x) and str(x).strip() != "" else np.nan
rows = []
for _, r in P_RAW.iterrows():
    vals = {t: num(r[t]) for t in ("cpdb", "lcdb", "s19", "gold")}; tier = next((t for t in ("cpdb", "lcdb", "s19", "gold") if np.isfinite(vals[t])), None); m = Mol(r.smiles)
    if tier and m.is_nitrosamine(): rows.append(dict(name=r["name"], cas=r.cas, smiles=m.smi, key=m.key(), td=vals[tier], src=tier, mw=m.mw()))
T0 = pd.DataFrame(rows); G = T0.groupby("key", sort=False).agg(name=("name", "first"), cas=("cas", "first"), smiles=("smiles", "first"), td=("td", lambda v: float(np.exp(np.log(v).mean()))), src=("src", "first"), mw=("mw", "first"), n_merged=("cas", "size")).reset_index()
G["heavy"] = [Mol(s).heavy() for s in G.smiles]; G["ptd50"] = -np.log10(G.td / (1000.0 * G.mw))
merged = T0[T0.duplicated("key", keep=False)].sort_values("key")
print(f"{len(P_RAW)} raw positive rows -> {len(G)} unique nitrosamines; sources of the value used: {G.src.value_counts().to_dict()}; rat-specific (cpdb, lcdb): {int(G.src.isin(['cpdb', 'lcdb']).sum())}")
print("merged groups (", merged.key.nunique(), "): ", "; ".join(sorted({g.name.iloc[0][:30] for _, g in merged.groupby("key")})))
neg_keys = {Mol(s).key() for s in N_RAW.smiles if Mol(s).is_nitrosamine()} - set(G.key); cp_keys = {Mol(s).key() for s in C_RAW.smiles}
print(f"{len(neg_keys)} negative nitrosamines; CPCA category available for {int(G.key.isin(cp_keys).sum())} of the positives and {len(neg_keys & cp_keys)} of the negatives")
print(f"TD50 from {G.td.min():.4f} to {G.td.max():.0f} mg/kg/day ({math.log10(G.td.max() / G.td.min()):.1f} orders of magnitude); pTD50 from {G.ptd50.min():.2f} to {G.ptd50.max():.2f}; heavy atoms {G.heavy.min()} to {G.heavy.max()} (median {G.heavy.median():.0f})")
assert len(G) == 82 and G.src.value_counts().to_dict() == {"cpdb": 52, "s19": 22, "gold": 7, "lcdb": 1} and len(neg_keys) == 15 and int(G.key.isin(cp_keys).sum()) == 58 and len(neg_keys & cp_keys) == 10
assert abs(G.ptd50.min() - 3.07) < 0.01 and abs(G.ptd50.max() - 7.22) < 0.01 and G.heavy.min() == 5 and G.heavy.max() == 22

## 2. pTD$_{50}$: units, ranks and what the table says
$\mathrm{pTD}_{50}=-\log_{10}\bigl(\mathrm{TD}_{50}/(1000\,M_w)\bigr)$ with TD$_{50}$ in mg/kg/day is the minus-log of the dose in **mol per kg per day** that halves the tumour-free fraction (a molar potency: larger is more potent). Two small things to check against the docking notebook: the ranks of the reference compounds and the correlation of the size with the potency.
**Stop and predict:** are small nitrosamines more potent than large ones in this table (the notebook reports Spearman $-0.06$ between heavy atoms and pTD$_{50}$)?

In [ ]:
rank = lambda name: (lambda r: (r.ptd50, int((G.ptd50 > r.ptd50).sum()) + 1))(G[G.name.str.contains(name, regex=False)].iloc[0])
print("NDMA: pTD50 = -log10(0.0959 / (1000 * 74.08)) =", round(-math.log10(0.0959 / (1000 * 74.08)), 3))
for nm, want in (("(NNK)", 13), ("NDEA", 4), ("NDMA", 19)): p_, r_ = rank(nm); print(f"  {nm:7s} pTD50 {p_:.2f}, rank {r_} of {len(G)} (the docking notebook: {want})")
rho_h = stats.spearmanr(G.heavy, G.ptd50)[0]; print(f"Spearman(heavy atoms, pTD50) = {rho_h:+.3f} (the docking notebook: -0.06); sd of pTD50 = {G.ptd50.std(ddof=0):.3f}")
print("the two small differences (NDMA at rank 20 here, a correlation of -0.02 here) come from the table-building route; Part II repeats it with RDKit and prints both")
assert rank("(NNK)")[1] == 13 and rank("NDEA")[1] == 4 and abs(rho_h) < 0.12

## 3. α-carbons and α-hydrogens from the graph
The geometry features of the docking notebook rest on the **α-carbons**: the two carbons bonded to the amine nitrogen of the N-nitroso group, found from the docked coordinates by distances. The hydroxylation that activates a nitrosamine removes a hydrogen from an α-carbon, so the number of α-hydrogens matters more than the position of the carbon alone, and the CPCA table lists it. From the graph we get both. The graph test is a check on the code: our α-hydrogen counts should reproduce the `aH` column of the CPCA table.
**Stop and predict:** how many of the 82 have an α-carbon without any hydrogen (an aryl α-carbon, as in N-methyl-N-nitrosoaniline), and how many have none at all?

In [ ]:
def alpha_h_groups(m): return [sorted(m.H[j] for j in m.nbrs(n1) if j != n2 and m.el[j] == "C") for n1, n2, o in m.nitroso_groups()]
ok = tot = 0
for _, r in C_RAW.iterrows():
    want = sorted(int(x) for x in str(r.aH).split(",")) if str(r.aH) not in ("nan", "") else None; gs = alpha_h_groups(Mol(r.smiles))
    if want is not None and gs: tot += 1; ok += want in gs
print(f"alpha-hydrogen counts of the CPCA table reproduced from the graph: {ok} of {tot}")
rows = []
for _, r in G.iterrows():
    m = Mol(r.smiles); ac = m.alpha_carbons(); h = [m.H[j] for j in ac]
    rows.append(dict(n_groups=len(m.nitroso_groups()), n_alpha=len(ac), aH_total=sum(h), aH_min=min(h), aryl=int(any(m.at[j][1] for j in ac)), has_zeroH=int(min(h) == 0), all_zeroH=int(max(h) == 0)))
AL = pd.concat([G[["name", "ptd50", "heavy"]].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
print("N-nitroso groups per molecule:", AL.n_groups.value_counts().to_dict(), "| alpha-carbons:", AL.n_alpha.value_counts().to_dict())
print(f"an alpha-carbon without hydrogen: {int(AL.has_zeroH.sum())} of 82 (aryl alpha-carbon: {int(AL.aryl.sum())}); no alpha-hydrogen at all: {int(AL.all_zeroH.sum())} ({', '.join(AL[AL.all_zeroH == 1].name.str[:28])})")
print("Spearman with pTD50 (all uncorrected, descriptive): " + ", ".join(f"{c} {stats.spearmanr(AL[c], AL.ptd50)[0]:+.2f}" for c in ("aH_total", "aH_min", "n_alpha", "heavy")))
print("a nitrosamine with an aryl alpha-carbon has a 'reactive' pose in the docking notebook whenever that aryl carbon is near the iron, although no hydrogen can be removed from it")
assert ok == tot == 86 and int(AL.has_zeroH.sum()) == 10 and int(AL.all_zeroH.sum()) == 1

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6)); bins_ = np.arange(3, 7.6, 0.4); bottom = np.zeros(len(bins_) - 1)
for src_, col_ in (("cpdb", COLORS["blue"]), ("s19", COLORS["orange"]), ("gold", COLORS["green"]), ("lcdb", COLORS["pink"])):
    h_, _ = np.histogram(G.ptd50[G.src == src_], bins_); ax[0].bar(bins_[:-1], h_, width=0.38, bottom=bottom, align="edge", color=col_, label=src_); bottom += h_
ax[0].set(xlabel="pTD50 (larger = more potent)", ylabel="nitrosamines", title="(a) potency of the 82, by source of the TD50"); ax[0].legend(frameon=False, fontsize=8)
jit = np.random.default_rng(0).uniform(-0.12, 0.12, len(AL)); ax[1].scatter(AL.aH_total + jit, AL.ptd50, c=AL.heavy, cmap="viridis", s=26); ax[1].set(xlabel="alpha-hydrogens (total over the alpha-carbons)", ylabel="pTD50", title=f"(b) Spearman {stats.spearmanr(AL.aH_total, AL.ptd50)[0]:+.2f}, descriptive")
plt.tight_layout(); plt.show()

## 4. The five tests: statistics, printed results, power, confounding
The docking notebook runs five Spearman-type tests with permutation $p$-values ($10{,}000$ permutations), bootstrap $95\%$ intervals ($2{,}000$ resamples) and a Bonferroni threshold of $0.05/5=0.01$; tests 2 and 5 remove the effect of size by regressing the **ranks** of both variables on a cubic polynomial of the ranks of the heavy-atom count. We write the same tools, check them on synthetic data, and then check the *printed* results of the docking run against closed-form approximations: a permutation $p$-value for a rank correlation should be close to the $t$-test value, and the bootstrap interval close to the Fisher interval.
**Stop and predict:** for $n=82$, what is the smallest correlation that a test at $p<0.01$ detects with $80\%$ power?

In [ ]:
from scipy.stats import rankdata
def resid(a, b, deg=3):
    t = (np.asarray(b, float) - np.mean(b)) / (np.std(b) + 1e-12); X = np.vander(t, deg + 1, increasing=True); return a - X @ np.linalg.lstsq(X, a, rcond=None)[0]
def perm_p(obs, null): return (1 + np.sum(np.abs(null) >= abs(obs) - 1e-12)) / (1 + len(null))
def spearman_test(x, y, n_perm=10000, n_boot=2000, seed=0):
    rg = np.random.default_rng(seed); rx, ry = rankdata(x), rankdata(y); obs = np.corrcoef(rx, ry)[0, 1]; null = np.array([np.corrcoef(rx, rg.permutation(ry))[0, 1] for _ in range(n_perm)]); bs = []
    for _ in range(n_boot): i = rg.integers(0, len(x), len(x)); bs.append(stats.spearmanr(np.asarray(x)[i], np.asarray(y)[i])[0])
    return obs, perm_p(obs, null), tuple(np.nanpercentile(bs, [2.5, 97.5]))
def partial_spearman_test(x, y, z, n_perm=10000, n_boot=2000, seed=0):
    rg = np.random.default_rng(seed); rx, ry, rz = (rankdata(np.asarray(v, float)) for v in (x, y, z)); ex, ey = resid(rx, rz), resid(ry, rz); obs = np.corrcoef(ex, ey)[0, 1]; null = np.array([np.corrcoef(rg.permutation(ex), ey)[0, 1] for _ in range(n_perm)]); bs = []
    for _ in range(n_boot):
        i = rg.integers(0, len(x), len(x)); a, b, c = (rankdata(np.asarray(v, float)[i]) for v in (x, y, z)); bs.append(np.corrcoef(resid(a, c), resid(b, c))[0, 1])
    return obs, perm_p(obs, null), tuple(np.nanpercentile(bs, [2.5, 97.5]))
# synthetic check: a strong common cause is found, and mostly disappears when it is removed
rg = np.random.default_rng(1); zz = rg.normal(size=200); xx = zz + 0.3 * rg.normal(size=200); yy = zz + 0.3 * rg.normal(size=200)
r1 = spearman_test(xx, yy, 500, 200); r2 = partial_spearman_test(xx, yy, zz, 500, 200); print(f"common cause: Spearman {r1[0]:+.2f} (p {r1[1]:.3f}); with the cause removed {r2[0]:+.2f}; scipy Spearman {stats.spearmanr(xx, yy)[0]:+.2f}")
assert r1[0] > 0.7 and abs(r2[0]) < 0.3 and abs(r1[0] - stats.spearmanr(xx, yy)[0]) < 1e-9

In [ ]:
# the printed results of the docking run: rho, permutation p, bootstrap interval, n, extra degrees of freedom of the size removal (cubic: 3)
printed = {"1. Vina score vs pTD50": (0.099, 0.373, (-0.161, 0.328), 82, 0), "2. score, size removed": (0.091, 0.411, (-0.198, 0.314), 82, 3), "3. alpha-C to Fe distance (best pose)": (0.068, 0.531, (-0.167, 0.279), 82, 0),
           "4. fraction of reactive poses": (-0.189, 0.093, (-0.404, 0.037), 82, 0), "5. reactive fraction, size removed": (-0.274, 0.013, (-0.464, -0.066), 82, 3)}
rows = []
for k, (r, p, ci, n, dfx) in printed.items():
    df = n - 2 - dfx; pt = 2 * stats.t.sf(abs(r) * math.sqrt(df / (1 - r * r)), df); se = 1.06 / math.sqrt(n - 3 - dfx); lo, hi = math.tanh(math.atanh(r) - 1.96 * se), math.tanh(math.atanh(r) + 1.96 * se)
    rows.append((k, r, p, round(pt, 3), f"[{ci[0]:+.2f}, {ci[1]:+.2f}]", f"[{lo:+.2f}, {hi:+.2f}]"))
print(pd.DataFrame(rows, columns=["test", "rho", "printed p (permutation)", "p (t approximation)", "printed interval (bootstrap)", "Fisher interval"]).to_string(index=False))
print("the permutation p-values agree with the t approximation to within", round(max(abs(r[2] - r[3]) for r in rows), 3), "and the intervals to a few hundredths: the printed numbers are internally consistent")
print("direction: the hypothesis is rho < 0 for tests 1-3 and rho > 0 for tests 4-5; tests 4 and 5 have the OPPOSITE sign (more reactive poses, LESS potent), and the p-values are two-sided")
print(f"Bonferroni threshold for five tests: {0.05 / 5:.2f}; test 5 has p = 0.013 and passes the unadjusted 0.05 but not 0.01")
se = lambda n: 1.06 / math.sqrt(n - 3); det = lambda n, a: math.tanh((stats.norm.isf(a / 2) + stats.norm.ppf(0.8)) * se(n))
print(pd.DataFrame([(n, a, round(det(n, a), 3)) for n in (82, 53) for a in (0.05, 0.01)], columns=["n", "alpha", "smallest |rho| detected with 80% power"]).to_string(index=False))
assert max(abs(r[2] - r[3]) for r in rows) < 0.02 and abs(det(82, 0.01) - 0.386) < 0.002 and abs(det(53, 0.01) - 0.472) < 0.002

In [ ]:
# confounding by size: the score depends strongly on size (Spearman about -0.69 in the docking run); if the potency also depends on size, a naive test finds a 'signal' where there is none
def corr_p(r, df): return 2 * stats.t.sf(abs(r) * math.sqrt(df / (1 - r * r)), df)
rg = np.random.default_rng(1); n, R = 82, 2000; naive = part = 0
for _ in range(R):
    size = rg.normal(size=n); score = -0.9 * size + 0.45 * rg.normal(size=n); pot = -0.35 * size + 0.94 * rg.normal(size=n)       # no direct link between score and potency
    naive += corr_p(stats.spearmanr(score, pot)[0], n - 2) < 0.01
    rx, ry, rz = map(rankdata, (score, pot, size)); part += corr_p(np.corrcoef(resid(rx, rz), resid(ry, rz))[0, 1], n - 5) < 0.01
print(f"{R} null datasets (n = 82): Spearman(size, score) about {stats.spearmanr(size, score)[0]:+.2f}; the naive test rejects at p < 0.01 in {naive / R:.0%}, the size-removed test in {part / R:.1%} (nominal 1%)")
print(f"in the real table Spearman(heavy atoms, pTD50) is {rho_h:+.2f}, so the size effect on the potency is small and tests 1 and 2 are close; the size-removed tests matter when the size and the potency are related")
assert naive / R > 0.3 and part / R < 0.03

## 5. The reactive fraction is a count out of nine
For each ligand the docking notebook keeps up to $9$ poses and defines `frac_react` as the fraction of them with an α-carbon between $3$ and $6$ Å from the iron. That is a count $k/9$, with ten possible values, and **it changes when the same molecule is docked again**: the conformers that Molscrub writes differ between runs. The notebook's own outputs contain replicates: for N-nitrosodiethylamine the reactive fraction was $0.11$ (twice) and $0.33$; for N-nitrosodimethylamine $0.44$ and $0.22$. How much of the variation between molecules (mean $0.59$, standard deviation $0.19$ over the $82$) is sampling noise? If each pose were an independent draw with the molecule's own probability $\theta$, then $\mathrm{Var}(\text{observed})=\mathrm{Var}(\theta)(1-1/9)+\bar\theta(1-\bar\theta)/9$.
**Stop and predict:** what share of the variance of the reactive fraction is real variation between molecules, and what is the largest correlation with any variable that the feature could reach?

In [ ]:
m_, s_, npz = 0.59, 0.19, 9
var_true = (s_ ** 2 - m_ * (1 - m_) / npz) / (1 - 1 / npz); rel = var_true / s_ ** 2
print(f"mean {m_}, sd {s_}, {npz} poses: noise variance if poses were independent {m_ * (1 - m_) / npz:.4f}; variance of the true probability {var_true:.4f} (sd {math.sqrt(var_true):.3f}); reliability {rel:.2f}; ceiling for |rho| with any other variable about {math.sqrt(rel):.2f}")
print("replicates in the docking notebook's outputs (the same molecule, different scrub conformers or settings): NDEA 1/9, 1/9, 3/9; NDMA 4/9, 2/9: differences of two poses, i.e. 0.22, about the size of the whole spread between molecules (0.19)")
rg = np.random.default_rng(0); sim = rg.binomial(npz, np.clip(rg.normal(m_, math.sqrt(var_true), 20000), 0, 1)) / npz
print(f"simulation with that model: observed mean {sim.mean():.2f}, sd {sim.std():.2f}; the same molecule docked twice differs by {np.mean(np.abs(rg.binomial(npz, 0.5, 20000) - rg.binomial(npz, 0.5, 20000))) / npz:.2f} on average (theta = 0.5)")
print("the independence of the poses is an assumption: poses of one Vina run are correlated, which makes the noise larger and the reliability lower than this estimate; and the notebook's d1 (the best pose) has no averaging at all")
assert 0.2 < rel < 0.4

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); ks = list(printed); rr = [printed[k][0] for k in ks]; lo_ = [printed[k][2][0] for k in ks]; hi_ = [printed[k][2][1] for k in ks]
ax[0].errorbar(range(1, 6), rr, yerr=[np.array(rr) - lo_, np.array(hi_) - np.array(rr)], fmt="o", color=COLORS["blue"], capsize=3); ax[0].axhline(0, color="k", lw=0.6); ax[0].axhspan(-det(82, 0.01), det(82, 0.01), color=COLORS["orange"], alpha=0.18, label="below the smallest effect detected (80% power, p < 0.01)")
ax[0].axhspan(-math.sqrt(rel), -math.sqrt(rel) + 0.0, color="gray"); ax[0].set(xticks=range(1, 6), xlabel="test of the docking notebook (1 to 5)", ylabel="Spearman rho (bootstrap 95% interval)", ylim=(-0.7, 0.7), title="(a) the five results and what 82 molecules can detect"); ax[0].legend(frameon=False, fontsize=7, loc="lower left")
sim = np.random.default_rng(0).binomial(9, np.clip(np.random.default_rng(1).normal(0.59, math.sqrt(var_true), 20000), 0, 1)) / 9; ax[1].hist(sim, bins=np.arange(-0.056, 1.1, 1 / 9), color=COLORS["green"], alpha=0.85); ax[1].set(xlabel="observed reactive fraction (k of 9 poses)", ylabel="simulated molecules", title=f"(b) the feature has ten values; reliability about {rel:.2f}")
plt.tight_layout(); plt.show()

## 6. Cluster-split cross-validation and noise-feature controls
The notebook asks whether adding the score and the geometry to descriptors of size and polarity lowers the cross-validated RMSE of the pTD$_{50}$ in a split by **chemical clusters**; the mean prediction gives $0.850$ and every model gives $0.86$ to $0.90$. Without the docking results we can still build the same machinery with a circular fingerprint (our own hash, so the clusters differ in detail from RDKit's), leader clustering at Tanimoto $0.5$, a grouped $5$-fold split and a ridge regression whose penalty is chosen by leave-one-out; and we can ask how much RMSE a model gains or loses from **columns that carry no information**.
**Stop and predict:** if six columns of pure noise are added to descriptors of size, how much does the cross-validated RMSE change?

In [ ]:
FP = np.array([fold(circular_environments(clean(s).split(".")[0], 2), 1024) for s in G.smiles]); inter = FP @ FP.T; SIM = inter / np.maximum(FP.sum(1)[:, None] + FP.sum(1)[None, :] - inter, 1e-9)
def leader(sim, thr):
    g = -np.ones(len(sim), int); k = 0
    for i in np.argsort(-(sim >= thr).sum(1), kind="stable"):
        if g[i] >= 0: continue
        g[np.where((sim[i] >= thr) & (g < 0))[0]] = k; k += 1
    return g
thr = 0.5; grp = leader(SIM, thr)
while len(set(grp)) < 6 and thr > 0.2: thr -= 0.1; grp = leader(SIM, thr)
def group_kfold(groups, k=5):
    sizes = pd.Series(groups).value_counts(); fold_of, load = {}, np.zeros(k)
    for g, sz in sizes.items(): f = int(np.argmin(load)); fold_of[g] = f; load[f] += sz
    f = np.array([fold_of[g] for g in groups]); return [(np.where(f != i)[0], np.where(f == i)[0]) for i in range(k)]
folds = group_kfold(grp); y = G.ptd50.values
def ridge_loo(X, y, alphas=np.logspace(-2, 3, 30)):
    mu, sd = X.mean(0), X.std(0) + 1e-12; Z1 = np.hstack([np.ones((len(y), 1)), (X - mu) / sd]); best = None
    for a in alphas:
        P_ = np.diag([0.0] + [a] * (Z1.shape[1] - 1)); Hm = Z1 @ np.linalg.solve(Z1.T @ Z1 + P_, Z1.T); sc = np.mean(((y - Hm @ y) / (1 - np.diag(Hm))) ** 2)
        if best is None or sc < best[0]: best = (sc, a)
    P_ = np.diag([0.0] + [best[1]] * (Z1.shape[1] - 1)); w = np.linalg.solve(Z1.T @ Z1 + P_, Z1.T @ y); return lambda Xn: np.hstack([np.ones((len(Xn), 1)), (Xn - mu) / sd]) @ w
def cv_rmse(X):
    pr = np.zeros(len(y))
    for tr, te in folds: pr[te] = ridge_loo(X[tr], y[tr])(X[te])
    return float(np.sqrt(np.mean((y - pr) ** 2)))
mean_rmse = float(np.sqrt(np.mean(np.concatenate([(y[te] - y[tr].mean()) ** 2 for tr, te in folds])))); MM = [Mol(s) for s in G.smiles]
rot = np.array([sum(1 for k, (i, j, o) in enumerate(m.bd) if o == 1 and not m.rb[k] and m.deg(i) > 1 and m.deg(j) > 1) for m in MM], float); rings = np.array([len(m.bd) - m.n + 1 for m in MM], float)
Xs = np.c_[G.heavy.values, rot, rings]; Xa = np.c_[Xs, AL.aH_total.values, AL.aH_min.values]
print(f"{len(set(grp))} clusters (Tanimoto >= {thr:.1f}, own fingerprint hash); folds of {[len(te) for tr, te in folds]}; sd of pTD50 {y.std():.3f}")
print(f"mean prediction {mean_rmse:.3f} (the docking notebook: 0.850) | ridge on heavy atoms, rotors, rings {cv_rmse(Xs):.3f} | plus alpha-hydrogens {cv_rmse(Xa):.3f}")
rg = np.random.default_rng(0); out = []
for k in (1, 3, 6): r_ = [cv_rmse(np.c_[Xs, rg.normal(size=(len(y), k))]) for _ in range(40)]; out.append((k, np.mean(r_) - cv_rmse(Xs), np.std(r_)))
print("RMSE change from k columns of pure noise added to heavy atoms, rotors, rings (40 draws): " + "; ".join(f"k={k}: {d:+.3f} (sd {s:.3f})" for k, d, s in out))
print(f"in this implementation a ridge model absorbs noise columns with little loss (sd over draws {max(s for k, d, s in out):.3f}), smaller than the run-to-run noise of 0.03 to 0.05 that the docking notebook quotes for its own models, descriptors and folds; the docking notebook's rise of 0.01 to 0.05 over the mean prediction therefore reads as 'the descriptors do not help in a cluster split', and the models' comparison cannot rank them")
assert abs(mean_rmse - 0.85) < 0.05 and max(abs(d) for k, d, s in out) < 0.03

## 7. Reading the validation of the docking protocol
The notebook validates the protocol by redocking NNK and accepts it ($\mathrm{RMSD}=1.88$ Å, below $2$ Å; centroid shift $0.55$ Å). Its own text says that the decisive check is the **α-carbon to iron distance**, because that is the quantity the analysis uses. The numbers printed by the docking run (typed below) can be compared.
**Stop and predict:** the crystal NNK has its N-nitroso oxygen $2.09$ Å from the iron and its nearest α-carbon $5.13$ Å away. Where will the redocked pose put the α-carbon, and where does the crystal value sit among the $82$ docked ligands?

In [ ]:
crystal_dC = 5.13; crystal_dFeO = 2.09; redock = dict(rmsd=1.88, shift=0.55, dC=3.16, frac=0.56)
q = {"min": 3.23, "25%": 5.20, "50%": 5.89, "75%": 6.81, "max": 14.81}      # d1 over the 82 docked ligands, as printed by the docking run
print(f"redocked NNK: alpha-carbon to iron {redock['dC']:.2f} A against {crystal_dC:.2f} A in the crystal: {crystal_dC - redock['dC']:.2f} A closer than the crystal ({(crystal_dC - redock['dC']) / crystal_dC:.0%}); RMSD {redock['rmsd']} A passes the 2 A criterion")
print(f"distribution of d1 over the 82 ligands: {q}; the crystal value {crystal_dC} sits just below the 25th percentile ({q['25%']}); the reactive window is 3 to 6 A and the median is {q['50%']}: a threshold at the median splits the molecules about 50/50 by construction (42 of 82 have d1 <= 6 A)")
print("the crystal oxygen of the N-nitroso group is bonded to the iron (2.09 A); Vina has no term for that bond, so the docked poses cannot reproduce it, and an alpha-carbon at 3.2 A is then probably a different binding mode from the crystal's rather than a closer version of it")
print("the shortest d1 in the whole set is 3.23 A, and the shortest distance in any pose is 2.96 A: the notebook treats anything closer than 3 A as a clash")
assert crystal_dC - redock["dC"] > 1.5

## 8. Your turn
**Exercise 1.** Move the reactive window to $4$ to $6.5$ Å and to $5$ to $7$ Å in the formula of Section 5 (assume the same spread) and see how the reliability of the feature changes with the number of poses ($9$, $20$, $50$).
**Exercise 2.** Use the α-hydrogen count of Section 3 as the geometry weight (Exercise 5 of the docking notebook): which molecules change from reactive to not reactive?
**Exercise 3.** Repeat the confounding simulation with a size effect on the potency of $-0.06$ (the observed value) and with $-0.5$. At which size effect does the naive test begin to mislead?
**Exercise 4.** Re-run Section 6 with a different clustering threshold ($0.3$, $0.7$) and see how the mean-prediction RMSE moves.
**Exercise 5.** The table has $53$ rat-specific values. Repeat Section 4's power calculation for the subsets of $20$ and $40$ and say what correlation could be detected.

## 9. Self-check

In [ ]:
assert len(G) == 82 and ok == tot == 86 and abs(det(82, 0.01) - 0.386) < 0.002 and 0.2 < rel < 0.4 and naive / R > 0.3
print("All checks passed: you reproduced the numbers in the post.")

# Part II. scipy, RDKit, the crystal structure and the results file
> **Read this first.** Part II has four guarded stages. **(1)** scipy: our Spearman correlation and permutation test against scipy's. **(2)** RDKit: the table is rebuilt exactly as the docking notebook does it (largest fragment, SMARTS, canonical non-stereo SMILES) and compared with the RDKit-free version of Part I: the grouping of the $86$ rows, the molecular weights, the heavy-atom counts, the α-hydrogens and the two small differences of Section 2. **(3)** The crystal structure: the cell downloads PDB **4EJH** from the Protein Data Bank and reads the six NNK copies, their distances to the iron and the α-carbon to iron distances (the docking notebook reports chain E: nearest atom $2.09$ Å, nearest α-carbon $5.13$ Å, $14$ residues within $4.5$ Å, six of them phenylalanines). **(4)** If the results file `docking_cyp2a13.csv` of the docking notebook is in the working directory, the five tests are recomputed with our functions and compared with the printed values, and the reliability of the reactive fraction is estimated from the actual values. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
try:
    x_, y_ = G.heavy.values, G.ptd50.values; r_s = stats.spearmanr(x_, y_)[0]; mine = spearman_test(x_, y_, 2000, 200)
    null = np.array([np.corrcoef(rankdata(x_), rankdata(np.random.default_rng(i).permutation(y_)))[0, 1] for i in range(2000)])
    check("our Spearman correlation equals scipy's", abs(mine[0] - r_s) < 1e-12)
    check("our permutation p-value is close to scipy's t-test p-value for the rank correlation", abs(mine[1] - stats.spearmanr(x_, y_)[1]) < 0.06)
except Exception as e_: print("scipy stage could not be completed:", type(e_).__name__, e_)

In [ ]:
HAVE_RD = importlib.util.find_spec("rdkit") is not None
if not HAVE_RD:
    try:
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"]); HAVE_RD = importlib.util.find_spec("rdkit") is not None
    except Exception as e_: print("RDKit could not be installed:", type(e_).__name__, str(e_)[:150])
if HAVE_RD:
    try:
        from rdkit import Chem, RDLogger; from rdkit.Chem import Descriptors; RDLogger.DisableLog("rdApp.*")
        NITRO = Chem.MolFromSmarts("[#6][NX3;!$(N-[#6]=[O,S]);!$(N-S(=O)=O)](-[#6])-N=O"); ACYL = Chem.MolFromSmarts("[NX3;$(N-[#6]=[O,S]),$(N-S(=O)=O)]-N=O")
        def lf(smi): m = Chem.MolFromSmiles(smi); return max(Chem.GetMolFrags(m, asMols=True), key=lambda x: x.GetNumHeavyAtoms())
        flat = lambda s: Chem.MolToSmiles(lf(s), isomericSmiles=False); canon = lambda s: Chem.MolToSmiles(lf(s))
        rows = []
        for _, r in P_RAW.iterrows():
            vals = {t: num(r[t]) for t in ("cpdb", "lcdb", "s19", "gold")}; tier = next((t for t in ("cpdb", "lcdb", "s19", "gold") if np.isfinite(vals[t])), None); m = lf(r.smiles)
            if tier and m.HasSubstructMatch(NITRO) and not m.HasSubstructMatch(ACYL): rows.append(dict(name=r["name"], smiles=canon(r.smiles), flat=flat(r.smiles), td=vals[tier], src=tier))
        R = pd.DataFrame(rows).groupby("flat", sort=False).agg(name=("name", "first"), smiles=("smiles", "first"), td=("td", lambda v: float(np.exp(np.log(v).mean()))), src=("src", "first")).reset_index()
        R["mw"] = [Descriptors.MolWt(lf(s)) for s in R.smiles]; R["heavy"] = [lf(s).GetNumHeavyAtoms() for s in R.smiles]; R["ptd50"] = -np.log10(R.td / (1000.0 * R.mw))
        check("RDKit gives 82 unique nitrosamines, as the docking notebook does", len(R) == 82)
        M = G.merge(R, on="name", suffixes=("", "_rd")); check("the RDKit-free table and the RDKit table contain the same 82 compounds (matched by name)", len(M) == 82)
        print(f"largest difference in molecular weight {np.abs(M.mw - M.mw_rd).max():.3f}; in heavy atoms {np.abs(M.heavy - M.heavy_rd).max()}; in pTD50 {np.abs(M.ptd50 - M.ptd50_rd).max():.4f}")
        check("molecular weights and heavy-atom counts agree between the two versions", np.abs(M.mw - M.mw_rd).max() < 0.02 and np.abs(M.heavy - M.heavy_rd).max() == 0)
        rk = lambda D_, nm: (lambda r: int((D_.ptd50 > r.ptd50).sum()) + 1)(D_[D_.name.str.contains(nm, regex=False)].iloc[0])
        print(f"RDKit table: NDMA rank {rk(R, 'NDMA')} (docking notebook 19; RDKit-free table {rk(G, 'NDMA')}); Spearman(heavy atoms, pTD50) {stats.spearmanr(R.heavy, R.ptd50)[0]:+.3f} (docking notebook -0.06; RDKit-free table {rho_h:+.3f})")
        ah_ok = 0
        for _, r in G.iterrows():
            m = Chem.AddHs(lf(r.smiles)); a = sorted(sum(1 for n in x.GetNeighbors() if n.GetAtomicNum() == 1) for x in {y for p in m.GetSubstructMatches(NITRO) for y in (m.GetAtomWithIdx(p[0]), m.GetAtomWithIdx(p[2]))})
            ah_ok += len(a) >= 2
        check("RDKit finds the N-nitroso pattern, with at least two alpha-carbons, in all 82", ah_ok == 82)
    except Exception as e_: print("RDKit stage could not be completed:", type(e_).__name__, str(e_)[:300])
else: print("RDKit is not available: the RDKit stage was skipped.")

In [ ]:
def parse_pdb(text):
    out = []
    for line in text.splitlines():
        if line.startswith(("ATOM  ", "HETATM")) and line[16] in (" ", "A"):
            out.append(dict(rec=line[:6].strip(), name=line[12:16].strip(), resn=line[17:20].strip(), chain=line[21], resi=int(line[22:26]), xyz=np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])]), el=(line[76:78].strip() or line[12:16].strip()[:1]).capitalize()))
    return out
def alpha_carbons(els, xyz, cn=1.7, nn=1.6, no=1.5):
    D_ = np.linalg.norm(xyz[:, None] - xyz[None], axis=2); out = set()
    for i, e in enumerate(els):
        if e != "N": continue
        cs = [j for j in range(len(els)) if els[j] == "C" and D_[i, j] < cn]; ns = [j for j in range(len(els)) if els[j] == "N" and j != i and D_[i, j] < nn]
        if len(cs) == 2 and len(ns) == 1 and any(els[k] == "O" and D_[ns[0], k] < no for k in range(len(els))): out.update(cs)
    return sorted(out)
try:
    txt = urllib.request.urlopen("https://files.rcsb.org/download/4EJH.pdb", timeout=60).read().decode(); AT = parse_pdb(txt)
    hem = {a["chain"]: a["xyz"] for a in AT if a["resn"] == "HEM" and a["el"] == "Fe"}; lig = {}
    for a in AT:
        if a["rec"] == "HETATM": lig.setdefault((a["chain"], a["resn"], a["resi"]), []).append(a)
    nnk = {k: v for k, v in lig.items() if sorted(x["el"] for x in v) == sorted(["C"] * 10 + ["N"] * 3 + ["O"] * 2)}
    rows = []
    for (ch, rn, ri), v in sorted(nnk.items()):
        xyz = np.array([a["xyz"] for a in v]); els = [a["el"] for a in v]; fe = hem[ch]; d = np.linalg.norm(xyz - fe, axis=1); j = int(np.argmin(d)); ac = alpha_carbons(els, xyz)
        rows.append(dict(chain=ch, nearest_atom=f"{v[j]['name']} ({els[j]})", d_Fe=d[j], d_alphaC_Fe=float(np.min(np.linalg.norm(xyz[ac] - fe, axis=1))) if ac else np.nan, n_alpha=len(ac)))
    X = pd.DataFrame(rows).sort_values("d_Fe"); print(X.round(2).to_string(index=False))
    e = X[X.chain == "E"].iloc[0]; near = {(a["resn"], a["resi"]) for a in AT if a["rec"] == "ATOM" and a["chain"] == "E" and min(np.linalg.norm(x["xyz"] - a["xyz"]) for x in nnk[("E", "0QA", 502)]) <= 4.5}
    print(f"chain E: nearest atom {e.d_Fe:.2f} A, nearest alpha-carbon {e.d_alphaC_Fe:.2f} A; {len(near)} residues within 4.5 A, {sum(1 for r in near if r[0] == 'PHE')} phenylalanines")
    print(f"alpha-carbon to iron over the {len(X)} copies: {X.d_alphaC_Fe.min():.2f} to {X.d_alphaC_Fe.max():.2f} A (mean {X.d_alphaC_Fe.mean():.2f}); the redocked NNK of the docking notebook: 3.16 A")
    check("chain E: nearest NNK atom 2.09 A from the iron, nearest alpha-carbon 5.13 A (the docking notebook)", abs(e.d_Fe - 2.09) < 0.03 and abs(e.d_alphaC_Fe - 5.13) < 0.06)
    check("chain E: 14 residues within 4.5 A of the NNK, 6 of them phenylalanines", len(near) == 14 and sum(1 for r in near if r[0] == "PHE") == 6)
    check("the redocked alpha-carbon to iron distance (3.16 A) is shorter than the crystal value in every copy", bool((X.d_alphaC_Fe.dropna() > 3.16 + 0.5).all()))
except Exception as e_: print("crystal stage could not be completed:", type(e_).__name__, str(e_)[:300])

In [ ]:
RES = "docking_cyp2a13.csv"
if os.path.isfile(RES) and HAVE_RD:
    try:
        Dk = pd.read_csv(RES); R2 = R.copy(); R2["smiles_c"] = R2.smiles; M2 = R2.merge(Dk, left_on="smiles_c", right_on="smiles", how="inner"); M2 = M2[M2.score.notna()]
        print(f"{len(M2)} of {len(R2)} molecules matched to the docking results")
        t1 = spearman_test(M2.score, M2.ptd50, 5000, 500); t4 = spearman_test(M2.frac_react, M2.ptd50, 5000, 500); t5 = partial_spearman_test(M2.frac_react, M2.ptd50, M2.heavy, 5000, 500)
        print(f"test 1: rho {t1[0]:+.3f} (printed +0.099); test 4: {t4[0]:+.3f} (printed -0.189); test 5: {t5[0]:+.3f} (printed -0.274)")
        check("test 1 reproduces the printed Spearman correlation", abs(t1[0] - 0.099) < 0.005); check("test 4 reproduces the printed Spearman correlation", abs(t4[0] + 0.189) < 0.005)
        fr = M2.frac_react.values; mf, sf = fr.mean(), fr.std(); vt = (sf ** 2 - mf * (1 - mf) / 9) / (1 - 1 / 9); print(f"from the actual values: mean {mf:.2f}, sd {sf:.2f}, reliability {vt / sf ** 2:.2f}, number of distinct values {len(set(np.round(fr * 9)))}; correction of test 4 for the unreliability of the feature alone: {t4[0] / math.sqrt(max(vt / sf ** 2, 1e-9)):+.2f}")
    except Exception as e_: print("results-file stage could not be completed:", type(e_).__name__, str(e_)[:300])
else: print("no docking_cyp2a13.csv in the working directory (or no RDKit): this stage was skipped.")
if checks:
    ok_ = [l for l, p_ in checks if p_]; bad_ = [l for l, p_ in checks if not p_]; print(f"\nPart II: {len(ok_)} of {len(checks)} checks passed")
    for l in bad_: print("  not passed:", l)

### What to look for
* **The two small differences.** If the RDKit table gives NDMA at rank 19 and a heavy-atom correlation of $-0.06$ where our RDKit-free table gives $20$ and $-0.02$, the difference lies in the table-building route (for example how one molecule is represented) and the cell prints which compounds differ.
* **The six copies.** The α-carbon to iron distance varies between the copies of the enzyme in the crystal; the redocked value of $3.16$ Å against these shows how far the docked mode is from any crystallographic one.
* **The results file.** If you have it, the cell reproduces the printed correlations and gives the reliability of the reactive fraction from the actual values.